# Multi-Expert NLI with POS Specialization
This model uses **spaCy** and **BERT** to create specialized experts for:
1. **Semantics** (Full context)
2. **Entities** (Nouns/Proper Nouns)
3. **Actions** (Verbs)
4. **Logic** (Dependency-based negation and lemma overlap)

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import spacy
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

# Load spaCy for linguistic analysis
try:
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
except:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["ner"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 31.8 MB/s eta 0:00:0031m33.6 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 1. Feature Engineering & POS Filtering
We define functions to strip sentences down to their core components (Nouns and Verbs) and extract logical features using dependency parsing.

In [3]:
def get_pos_filtered_text(text, pos_tags):
    doc = nlp(str(text))
    tokens = [token.text for token in doc if token.pos_ in pos_tags]
    return " ".join(tokens) if tokens else "none"

def extract_logic_features(premise, hypothesis):
    p_doc = nlp(str(premise))
    h_doc = nlp(str(hypothesis))
    
    # Dependency-based negation detection
    neg_p = sum(1 for t in p_doc if t.dep_ == "neg")
    neg_h = sum(1 for t in h_doc if t.dep_ == "neg")
    
    # Lemma-based overlap (more robust than raw strings)
    p_set = {t.lemma_.lower() for t in p_doc if not t.is_stop and not t.is_punct}
    h_set = {t.lemma_.lower() for t in h_doc if not t.is_stop and not t.is_punct}
    
    overlap = len(p_set & h_set) / len(p_set | h_set) if (p_set | h_set) else 0.0
    
    return [float(neg_p), float(neg_h), float(abs(neg_p - neg_h)), float(overlap)]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 128

def preprocess_function(example):
    # Main Semantic input
    main_enc = tokenizer(example["premise"], example["hypothesis"], 
                         truncation=True, padding="max_length", max_length=MAX_LEN)
    
    # POS Filtering for Specialized Experts
    p_nouns = get_pos_filtered_text(example["premise"], ["NOUN", "PROPN"])
    h_nouns = get_pos_filtered_text(example["hypothesis"], ["NOUN", "PROPN"])
    p_verbs = get_pos_filtered_text(example["premise"], ["VERB"])
    h_verbs = get_pos_filtered_text(example["hypothesis"], ["VERB"])

    # Specialized encodings
    entity_enc = tokenizer(p_nouns, h_nouns, truncation=True, padding="max_length", max_length=MAX_LEN)
    action_enc = tokenizer(p_verbs, h_verbs, truncation=True, padding="max_length", max_length=MAX_LEN)

    return {
        "input_ids": main_enc["input_ids"],
        "attention_mask": main_enc["attention_mask"],
        "entity_ids": entity_enc["input_ids"],
        "action_ids": action_enc["input_ids"],
        "logic_features": extract_logic_features(example["premise"], example["hypothesis"]),
        "label": int(example["label"])
    }

## 2. Model Architecture
The model incorporates a Shared BERT encoder but routes specific information through four distinct expert heads. The **Gating Network** uses the global context to decide the importance of each expert.

In [4]:
class POSSpecializedNLI(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h_size = self.encoder.config.hidden_size

        # Expert heads
        self.semantic_expert = nn.Linear(h_size, num_labels)
        self.entity_expert = nn.Linear(h_size, num_labels)
        self.action_expert = nn.Linear(h_size, num_labels)
        self.logic_expert = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, num_labels)
        )

        # Gating network: chooses between 4 experts
        self.gating = nn.Sequential(
            nn.Linear(h_size + 4, 64), 
            nn.ReLU(), 
            nn.Linear(64, 4)
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, entity_ids, action_ids, logic_features, labels=None):
        # Get BERT pooler outputs (CLS tokens) for all three views
        sem_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        ent_out = self.encoder(input_ids=entity_ids).last_hidden_state[:, 0, :]
        act_out = self.encoder(input_ids=action_ids).last_hidden_state[:, 0, :]

        # Compute logits from each expert
        l_sem = self.semantic_expert(sem_out)
        l_ent = self.entity_expert(ent_out)
        l_act = self.action_expert(act_out)
        l_log = self.logic_expert(logic_features.float())

        # Determine gating weights
        gate_input = torch.cat([sem_out, logic_features.float()], dim=1)
        gate_weights = torch.softmax(self.gating(gate_input), dim=1)

        # Mixture of Experts fusion
        final_logits = (gate_weights[:, 0:1] * l_sem + 
                        gate_weights[:, 1:2] * l_ent + 
                        gate_weights[:, 2:3] * l_act +
                        gate_weights[:, 3:4] * l_log)

        loss = self.loss_fn(final_logits, labels) if labels is not None else None
        return {"loss": loss, "logits": final_logits}

## 3. Training & Execution
Update the paths below to point to your CSV files.

In [5]:
# Load and Map Data
train_df = pd.read_csv("training_data/NLI/train.csv")
dev_df = pd.read_csv("training_data/NLI/dev.csv")

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function)
dev_dataset = Dataset.from_pandas(dev_df).map(preprocess_function)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "entity_ids", "action_ids", "logic_features", "label"])
dev_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "entity_ids", "action_ids", "logic_features", "label"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds), 
        "macro_f1": f1_score(labels, preds, average="macro")
    }

model = POSSpecializedNLI()

training_args = TrainingArguments(
    output_dir="results_pos_expert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.431395,0.419370,0.811609,0.811592
2,0.280954,0.470805,0.826900,0.826075
3,0.162072,0.707657,0.829424,0.829191


RuntimeError: on_train_begin must be called before on_evaluate